In [0]:
# ============================================================
# ONLINE PICKING - ATTRIBUTE DECODING AND THE ZONE SPLIT
#
# Source filter is area only:
#
#   {"name": "D.Analysis - Online Picking - <zone group>",
#    "area": "('Online Picking')", "event": None}
#
# so no event filter can hide anything. What this tests is everything that
# happens AFTER the rows are found:
#
#   1. location:XH117B                        -> "Zone X, Aisle H"
#   2. location:VL001C, lastLocation:VK162F   -> "Zone V, Aisle L; Last location: Zone V, Aisle K"
#   3. no location attribute at all           -> "" (and no zone, so no report claims it)
#   4. the zone letter then picks the report: Drive / Way / E3
#
# READ ONLY - nothing here touches the Google Sheet.
# ============================================================

dbutils.widgets.text("on_date", "12/08/2026", "Date (dd/MM/yyyy)")
dbutils.widgets.text("from_time", "03:15", "From Time (HH:mm, UK local)")
dbutils.widgets.text("to_time", "03:30", "To Time (HH:mm, UK local)")
dbutils.widgets.text("warehouse", "X", "Warehouse Code")
dbutils.widgets.text("area", "Online Picking", "Area Code")

from datetime import datetime
from zoneinfo import ZoneInfo

on_date = dbutils.widgets.get("on_date").strip()
from_time = dbutils.widgets.get("from_time").strip()
to_time = dbutils.widgets.get("to_time").strip()
warehouse = dbutils.widgets.get("warehouse").strip()
area = dbutils.widgets.get("area").strip()

uk = ZoneInfo("Europe/London")
start_local = datetime.strptime(on_date + " " + from_time, "%d/%m/%Y %H:%M").replace(tzinfo=uk)
end_local = datetime.strptime(on_date + " " + to_time, "%d/%m/%Y %H:%M").replace(tzinfo=uk)
if end_local <= start_local:
    raise ValueError("To time must be after From time")

start_utc = start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")
end_utc = end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S")


def _week_val(date_local):
    """Same retail week the live notebook writes, so Week matches the Data tab."""
    epoch_ref = datetime(2025, 12, 14).date()
    return ((date_local - epoch_ref).days // 7 + 46) % 52


week_val = _week_val(start_local.date())
date_time_range = (start_local.strftime("%d/%m/%Y %H:%M") + " - " +
                   end_local.strftime("%d/%m/%Y %H:%M"))

ZONES = {
    "Drive": ["D", "F", "Q", "S", "T", "V"],
    "Way":   ["A", "B", "E", "G", "J", "L", "X", "W", "R"],
    "E3":    ["H", "C"],
}

print("Window (UK local) :", date_time_range)
print("Window (UTC)      :", start_utc, "->", end_utc)
print("Warehouse / area  :", repr(warehouse), "/", repr(area))
print("Week              :", week_val)
for g, z in ZONES.items():
    print(f"  {g:<6} zones: {' '.join(z)}")

In [0]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))

df.createOrReplaceTempView("landing_bonus_hub_event_parsed")
print("View created.")

In [0]:
# ------------------------------------------------------------
# Matching is CASE-INSENSITIVE and does not assume a space after the colon.
# The real values are "location:XH117B" - lower case, no space - so a pattern
# of 'Location: %' matches nothing at all. That is worth being deliberate
# about, because a near-miss here returns zero rows rather than wrong ones.
#
# 'lastLocation:...' does not start with 'location:', so the two keys cannot
# be confused by a prefix test.
# ------------------------------------------------------------

def _attr(key):
    """First attribute whose key is `key`, or NULL. try_element_at, not
    element_at: element_at raises on an EMPTY array under ANSI mode, and an
    event with no such attribute filters down to exactly that."""
    return ("try_element_at(filter(PAYLOAD_ATTRIBUTES, "
            f"x -> lower(x) LIKE '{key.lower()}:%'), 1)")


def _code(attr):
    """'location:XH117B' -> 'XH117B'. Everything up to the first colon goes,
    whatever the key was called, and any spacing after it is trimmed."""
    return f"trim(regexp_replace({attr}, '^[^:]*:', ''))"


def _zone_aisle(attr):
    """'location:XH117B' -> 'Zone X, Aisle H'. First character is the zone,
    second the aisle; bay and level are dropped. Anything too short to hold
    both, or absent entirely, gives '' rather than a half-formed label."""
    c = _code(attr)
    return (f"CASE WHEN {c} IS NULL OR length({c}) < 2 THEN '' "
            f"ELSE concat('Zone ', upper(substring({c}, 1, 1)), "
            f"', Aisle ', upper(substring({c}, 2, 1))) END")


LOC, LAST = _attr("location"), _attr("lastLocation")
ZA, ZA_LAST = _zone_aisle(LOC), _zone_aisle(LAST)

# The Attribute column. A last location is appended when the event carries one
# - keyed off the attribute being present rather than off the event being
# named ChangeAisleEvent, so any other event carrying one is handled too.
ATTR_SQL = (f"CASE WHEN {ZA} = '' THEN '' "
            f"WHEN {ZA_LAST} = '' THEN {ZA} "
            f"ELSE concat({ZA}, '; Last location: ', {ZA_LAST}) END")

# Zone letter on its own, for the report split.
_C = _code(LOC)
ZONE_SQL = (f"CASE WHEN {_C} IS NULL OR length({_C}) < 1 THEN '' "
            f"ELSE upper(substring({_C}, 1, 1)) END")

_when = " ".join(
    f"""WHEN {ZONE_SQL} IN ({", ".join(f"'{z}'" for z in zs)}) """
    f"""THEN 'D.Analysis - Online Picking - {g}'"""
    for g, zs in ZONES.items())
REPORT_SQL = f"CASE {_when} ELSE NULL END"

WHERE_SQL = f"""TRIM(PAYLOAD_WAREHOUSECODE) = '{warehouse}'
    AND PAYLOAD_AREACODE IN ('{area}')
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{start_utc}')
    AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{end_utc}')"""

print("Attribute :", ATTR_SQL[:110], "...")
print("Zone      :", ZONE_SQL[:110], "...")
print("Report    :", REPORT_SQL[:110], "...")

In [0]:
# The decode beside its input, so it can be judged against the source rather
# than trusted. Every distinct attribute array in the window appears once.

display(spark.sql(f"""
  SELECT
    PAYLOAD_ATTRIBUTES   AS attributes_raw,
    {LOC}                AS location_attr,
    {LAST}               AS last_location_attr,
    {ATTR_SQL}           AS `Attribute (decoded)`,
    {ZONE_SQL}           AS zone,
    COALESCE({REPORT_SQL}, '(none - DROPPED)') AS goes_to_report,
    PAYLOAD_EVENTTYPE    AS event_type,
    COUNT(*)             AS events
  FROM landing_bonus_hub_event_parsed
  WHERE {WHERE_SQL}
  GROUP BY 1, 2, 3, 4, 5, 6, 7
  ORDER BY goes_to_report, zone, `Attribute (decoded)`
"""))

In [0]:
# The three reports as they would actually be written: grouped by date, hour,
# bonus, event type and the decoded Attribute, split across Drive / Way / E3
# by zone.
#
# Events with no location have no zone, so no report claims them and they are
# dropped here. The next cell reports what that costs, which is the number to
# watch: standard hours dropped are hours an operator worked and is not
# credited for.

data_tab_rows = spark.sql(f"""
  SELECT
    date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'dd/MM/yyyy') AS `Date`,
    hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'))                     AS `Hour`,
    upper(trim(PAYLOAD_BONUSCODE))                                                                     AS `PAYLOAD_BONUSCODE`,
    PAYLOAD_EVENTTYPE                                                                                  AS `PAYLOAD_EVENTTYPE`,
    {ATTR_SQL}                                                                                         AS `Attribute`,
    SUM(PAYLOAD_QUANTITY)                                                                              AS `Total_Quantity`,
    ROUND(SUM(PAYLOAD_STANDARDHOURS), 4)                                                               AS `Total_StandardHours`,
    ROUND(SUM(PAYLOAD_SMV), 2)                                                                         AS `Total_SMV`,
    {week_val}                                                                                         AS `Week`,
    '{date_time_range}'                                                                                AS `Date Time Range`,
    {REPORT_SQL}                                                                                       AS `Report Name`
  FROM landing_bonus_hub_event_parsed
  WHERE {WHERE_SQL}
    AND {REPORT_SQL} IS NOT NULL
  GROUP BY 1, 2, 3, 4, 5, 11
  ORDER BY `Report Name`, `PAYLOAD_BONUSCODE`, `PAYLOAD_EVENTTYPE`
""")

print(f"{data_tab_rows.count()} row(s) would be written for this 15-minute window")
display(data_tab_rows)

In [0]:
# An event with no location has no zone, so no report claims it. That is only
# harmless if those events carry no standard hours - the dashboard's
# productivity figure is standard hours over head deployed, so hours lost here
# are hours the operator worked and is not credited for.
#
# Read the `std_hours` column: a non-zero total against "(none - DROPPED)" is
# a real problem and needs those events assigned somewhere.

display(spark.sql(f"""
  SELECT
    COALESCE({REPORT_SQL}, '(none - DROPPED)') AS goes_to_report,
    PAYLOAD_EVENTTYPE                          AS event_type,
    COUNT(*)                                   AS events,
    SUM(PAYLOAD_QUANTITY)                      AS qty,
    ROUND(SUM(PAYLOAD_STANDARDHOURS), 4)       AS std_hours,
    ROUND(SUM(PAYLOAD_SMV), 2)                 AS smv
  FROM landing_bonus_hub_event_parsed
  WHERE {WHERE_SQL}
  GROUP BY 1, 2
  ORDER BY goes_to_report, std_hours DESC
"""))

In [0]:
kept, dropped = spark.sql(f"""
  SELECT
    SUM(CASE WHEN {REPORT_SQL} IS NOT NULL THEN PAYLOAD_STANDARDHOURS ELSE 0 END) AS kept,
    SUM(CASE WHEN {REPORT_SQL} IS NULL     THEN PAYLOAD_STANDARDHOURS ELSE 0 END) AS dropped
  FROM landing_bonus_hub_event_parsed
  WHERE {WHERE_SQL}
""").collect()[0]

kept = float(kept or 0)
dropped = float(dropped or 0)
total = kept + dropped
print(f"standard hours claimed by a report : {kept:.4f}")
print(f"standard hours dropped             : {dropped:.4f}")
if total:
    print(f"dropped share                      : {dropped / total:.2%}")

display(spark.sql(f"""
  SELECT
    {REPORT_SQL}                                   AS report,
    COUNT(*)                                       AS events,
    COUNT(DISTINCT upper(trim(PAYLOAD_BONUSCODE))) AS operators,
    COUNT(DISTINCT {ATTR_SQL})                     AS distinct_attributes,
    SUM(PAYLOAD_QUANTITY)                          AS qty,
    ROUND(SUM(PAYLOAD_STANDARDHOURS), 4)           AS std_hours
  FROM landing_bonus_hub_event_parsed
  WHERE {WHERE_SQL} AND {REPORT_SQL} IS NOT NULL
  GROUP BY 1
  ORDER BY std_hours DESC
"""))

In [0]:
# Any letter here that is in none of the three sets is claimed by no report.

zones = spark.sql(f"""
  SELECT {ZONE_SQL} AS zone, COUNT(*) AS events,
         ROUND(SUM(PAYLOAD_STANDARDHOURS), 4) AS std_hours
  FROM landing_bonus_hub_event_parsed
  WHERE {WHERE_SQL}
  GROUP BY 1
  ORDER BY events DESC
""").collect()

owner_of = {z: g for g, zs in ZONES.items() for z in zs}
print(f"{'zone':<8}{'events':>9}{'std hours':>12}   claimed by")
print("-" * 52)
for r in zones:
    z = (r["zone"] or "").strip()
    owner = owner_of.get(z, "(none - DROPPED)" if z else "(no location on event)")
    print(f"{z or '-':<8}{r['events']:>9}{float(r['std_hours'] or 0):>12.4f}   {owner}")

In [0]:
display(spark.sql(f"""
  SELECT
    from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP), 'Europe/London') AS event_time_uk,
    upper(trim(PAYLOAD_BONUSCODE)) AS bonus,
    PAYLOAD_EVENTTYPE              AS event_type,
    PAYLOAD_ATTRIBUTES             AS attributes_raw,
    {ATTR_SQL}                     AS attribute_decoded,
    {ZONE_SQL}                     AS zone,
    COALESCE({REPORT_SQL}, '(none - DROPPED)') AS goes_to_report,
    PAYLOAD_QUANTITY               AS qty,
    PAYLOAD_STANDARDHOURS          AS std_hours,
    PAYLOAD_SMV                    AS smv
  FROM landing_bonus_hub_event_parsed
  WHERE {WHERE_SQL}
  ORDER BY bonus, event_time_uk
"""))